## Document Parsing with Qwen3-VL

Welcome to this notebook, which showcases the powerful document parsing capabilities of our model. It can process any image and output its contents in various formats such as HTML, JSON, Markdown, and LaTeX. Notably, we introduce two unique Qwenvl formats:

- Qwenvl HTML format, which adds positional information for each component to enable precise document reconstruction and manipulation.
- Qwenvl Markdown format, which converts the overall image content into Markdown. In this format, all tables are represented in LaTeX with their corresponding coordinates indicated before each table, and images are replaced with coordinate-based placeholders for accurate positioning.

This allows for highly detailed and flexible document parsing and reconstruction.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
# Required: OPENAI_API_KEY, OPENAI_BASE_URL, MODEL_ID
load_dotenv()

# Set OpenAI API credentials (required for OpenAI-compatible server)
API_KEY = os.environ["OPENAI_API_KEY"]
BASE_URL = os.environ["OPENAI_BASE_URL"]
MODEL_ID = os.environ["MODEL_ID"]

print("✓ Environment variables loaded successfully")
print(f"  API_KEY: {API_KEY[:8]}..." if API_KEY else "  API_KEY: (empty)")
print(f"  BASE_URL: {BASE_URL}")
print(f"  MODEL_ID: {MODEL_ID}")


#### \[Setup\]

Load visualization utils.

In [ ]:
!pip install git+https://github.com/huggingface/transformers
!pip install qwen-vl-utils
!pip install qwen_agent
!pip install openai

In [ ]:
# Get Noto font
#!apt-get install fonts-noto-cjk 

import requests
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
from bs4 import BeautifulSoup, Tag
import re

def smart_resize(height, width, factor=32, min_pixels=65536, max_pixels=2457600):
    if height < factor or width < factor:
        raise ValueError(f"La imagen es demasiado pequeña: {height}x{width}")
    
    # Calcular escala para mantenerse dentro de los límites de píxeles
    h_w_ratio = height / width
    target_pixels = max(min(height * width, max_pixels), min_pixels)
    
    new_h = int((target_pixels * h_w_ratio)**0.5)
    new_w = int((target_pixels / h_w_ratio)**0.5)
    
    # Ajustar al múltiplo más cercano del factor (32 para Qwen3.5)
    new_h = max(round(new_h / factor) * factor, factor)
    new_w = max(round(new_w / factor) * factor, factor)
    
    return new_h, new_w

# Function to draw bounding boxes and text on images based on HTML content
def draw_bbox_html(image_path, full_predict):
    """
    可视化 Qwenvl HTML 的 data-bbox 框并展示文本，坐标为相对 0-1000。
    过滤规则：跳过 <ol>，仅绘制 <li> 子项和其它元素。
    """
    # 读取图片
    if image_path.startswith("http"):
        response = requests.get(image_path)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_path).convert("RGB")
    width = image.width
    height = image.height

    soup = BeautifulSoup(full_predict, 'html.parser')
    elements_with_bbox = soup.find_all(attrs={'data-bbox': True})

    # 保留原过滤逻辑
    filtered_elements = []
    for el in elements_with_bbox:
        if el.name == 'ol':
            continue  # 跳过 <ol>
        elif el.name == 'li' and el.parent.name == 'ol':
            filtered_elements.append(el)  # 仅保留 <ol> 下的 <li>
        else:
            filtered_elements.append(el)

    # 字体兼容
    try:
        font = ImageFont.truetype("NotoSansCJK-Regular.ttc", 10)
    except Exception:
        font = ImageFont.load_default()
    draw = ImageDraw.Draw(image)
    
    # 绘制框与文本
    for element in filtered_elements:
        bbox_str = element['data-bbox']
        text = element.get_text(strip=True)
        try:
            x1, y1, x2, y2 = map(int, bbox_str.split())
        except Exception:
            continue

        bx1 = int(x1 / 1000 * width)
        by1 = int(y1 / 1000 * height)
        bx2 = int(x2 / 1000 * width)
        by2 = int(y2 / 1000 * height)
        
        if bx1 > bx2:
            bx1, bx2 = bx2, bx1
        if by1 > by2:
            by1, by2 = by2, by1
            
        draw.rectangle([bx1, by1, bx2, by2], outline='red', width=2)
        draw.text((bx1, by2), text, fill='black', font=font)

    image.show()

    
# Function to draw bounding boxes on images based on Markdown content
def draw_bbox_markdown(image_path, md_content):
    """
    只可视化Markdown中的 <!-- Image/Table (x1, y1, x2, y2) --> 坐标框，坐标为相对0-1000
    Table 用绿色框，Image 用蓝色框。
    """
    if image_path.startswith("http"):
        response = requests.get(image_path)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_path).convert("RGB")
    width = image.width
    height = image.height

    pattern = r"<!-- (Image|Table) \(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\) -->"
    matches = re.findall(pattern, md_content)
    draw = ImageDraw.Draw(image)
    for item in matches:
        typ, x1, y1, x2, y2 = item
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        bx1 = int(x1 / 1000 * width)
        by1 = int(y1 / 1000 * height)
        bx2 = int(x2 / 1000 * width)
        by2 = int(y2 / 1000 * height)
        if bx1 > bx2:
            bx1, bx2 = bx2, bx1
        if by1 > by2:
            by1, by2 = by2, by1
        color = 'blue' if typ == "Image" else 'red'
        draw.rectangle([bx1, by1, bx2, by2], outline=color, width=6)

    image.show()


# Function to clean and format HTML content
def clean_and_format_html(full_predict):
    soup = BeautifulSoup(full_predict, 'html.parser')
    
    # Regular expression pattern to match 'color' styles in style attributes
    color_pattern = re.compile(r'\bcolor:[^;]+;?')

    # Find all tags with style attributes and remove 'color' styles
    for tag in soup.find_all(style=True):
        original_style = tag.get('style', '')
        new_style = color_pattern.sub('', original_style)
        if not new_style.strip():
            del tag['style']
        else:
            new_style = new_style.rstrip(';')
            tag['style'] = new_style
            
    # Remove 'data-bbox' and 'data-polygon' attributes from all tags
    for attr in ["data-bbox", "data-polygon"]:
        for tag in soup.find_all(attrs={attr: True}):
            del tag[attr]

    classes_to_update = ['formula.machine_printed', 'formula.handwritten']
    # Update specific class names in div tags
    for tag in soup.find_all(class_=True):
        if isinstance(tag, Tag) and 'class' in tag.attrs:
            new_classes = [cls if cls not in classes_to_update else 'formula' for cls in tag.get('class', [])]
            tag['class'] = list(dict.fromkeys(new_classes))  # Deduplicate and update class names

    # Clear contents of divs with specific class names and rename their classes
    for div in soup.find_all('div', class_='image caption'):
        div.clear()
        div['class'] = ['image']

    classes_to_clean = ['music sheet', 'chemical formula', 'chart']
    # Clear contents and remove 'format' attributes of tags with specific class names
    for class_name in classes_to_clean:
        for tag in soup.find_all(class_=class_name):
            if isinstance(tag, Tag):
                tag.clear()
                if 'format' in tag.attrs:
                    del tag['format']

    # Manually build the output string
    output = []
    for child in soup.body.children:
        if isinstance(child, Tag):
            output.append(str(child))
            output.append('\n')  # Add newline after each top-level element
        elif isinstance(child, str) and not child.strip():
            continue  # Ignore whitespace text nodes
    complete_html = f"""```html\n<html><body>\n{" ".join(output)}</body></html>\n```"""
    return complete_html

inference function with API

In [ ]:
from openai import OpenAI
import os
import base64
#  base 64 编码格式
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


# @title inference function with API
def inference_with_api(image_path, prompt, model_id=MODEL_ID, min_pixels=512*32*32, max_pixels=2048*32*32):
    base64_image = encode_image(image_path)
    client = OpenAI(
        #If the environment variable is not configured, please replace the following line with the Dashscope API Key: api_key="sk-xxx".
        api_key=API_KEY,
        base_url=BASE_URL
    )


    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "min_pixels": min_pixels,
                    "max_pixels": max_pixels,
                    # Pass in BASE64 image data. Note that the image format (i.e., image/{format}) must match the Content Type in the list of supported images. "f" is the method for string formatting.
                    # PNG image:  f"data:image/png;base64,{base64_image}"
                    # JPEG image: f"data:image/jpeg;base64,{base64_image}"
                    # WEBP image: f"data:image/webp;base64,{base64_image}"
                    "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]
    completion = client.chat.completions.create(
        model = model_id,
        messages = messages,
       
    )
    return completion.choices[0].message.content

#### 1. Document Parsing in QwenVL HTML Format

Here shows how to generate and process HTML content using Qwen2.5-VL. The generated HTML follows the QwenVL Document Parser format with bounding boxes.

In [ ]:
import requests
from io import BytesIO
import os

img_url = "https://ofasys-multimodal-wlcb-3-toshanghai.oss-cn-shanghai.aliyuncs.com/Qwen3VL/demo/omni_parsing/179729.jpg"
response = requests.get(img_url)
img_name = os.path.basename(img_url)
image = Image.open(BytesIO(response.content))
image.save(img_name)

prompt = "qwenvl html"

min_pixels = 512*32*32
max_pixels = 2048*32*32
width, height = image.size
input_height, input_width = smart_resize(height, width, min_pixels=min_pixels, max_pixels=max_pixels, factor=32)
output = inference_with_api(img_name, prompt, min_pixels=min_pixels, max_pixels=max_pixels)


# Visualization
print(input_height, input_width)
print(output)
draw_bbox_html(img_url, output)

ordinary_html = clean_and_format_html(output)
print(ordinary_html)

#### 2. Document Parsing in QwenVL Markdown Format

Here shows how to generate and process Markdown content using Qwen3-VL. The generated Markdown follows the QwenVL Markdown format, including positional information. In this format, all detected tables are represented as LaTeX with their corresponding coordinates indicated before each table, and images are replaced with coordinate-based placeholders for accurate positioning.

In [ ]:
import requests
from io import BytesIO
import os

img_url = "https://ofasys-multimodal-wlcb-3-toshanghai.oss-cn-shanghai.aliyuncs.com/Qwen3VL/demo/omni_parsing/120922.jpg"
response = requests.get(img_url)
img_name = os.path.basename(img_url)
image = Image.open(BytesIO(response.content))
image.save(img_name)

prompt = "qwenvl markdown"

min_pixels = 512*32*32
max_pixels = 4608*32*32
width, height = image.size
input_height, input_width = smart_resize(height, width, min_pixels=min_pixels, max_pixels=max_pixels, factor=32)
output = inference_with_api(img_name, prompt, min_pixels=min_pixels, max_pixels=max_pixels)


# Visualization
print(input_height, input_width)
print(output)
draw_bbox_markdown(img_url, output)
